# Clase 5 - EPG4002
## Mean Shift, DBSCAN y visualización con componentes principales en R

Este notebook replica la clase en Python usando código R para JupyterLab. Se consideran:

- un ejemplo pequeño en dos dimensiones;
- el algoritmo **Mean Shift**, usando el paquete `MeanShift` y la función `msClustering()`;
- el algoritmo **DBSCAN**, usando el paquete `dbscan`;
- una aplicación a la base `iris`;
- visualización mediante componentes principales.

En los scripts previos del curso se utiliza `MeanShift::msClustering()`, cuya entrada espera los datos transpuestos. Por eso, si la matriz de datos tiene individuos en filas y variables en columnas, se usa `t(X)`.


In [ ]:
# Configuración inicial
options(repr.plot.width = 7, repr.plot.height = 5)
set.seed(123)

# Instalar paquetes si no están disponibles
if (!requireNamespace("MeanShift", quietly = TRUE)) {
  install.packages("MeanShift")
}

if (!requireNamespace("dbscan", quietly = TRUE)) {
  install.packages("dbscan")
}

if (!requireNamespace("MASS", quietly = TRUE)) {
  install.packages("MASS")
}

library(MeanShift)
library(dbscan)
library(MASS)


## 1. Datos de ejemplo

Se comienza con un conjunto pequeño de observaciones en dos dimensiones. Además, se agregan tres observaciones nuevas para ilustrar la asignación posterior en Mean Shift.

In [ ]:
# Datos de ejemplo de clases
df <- data.frame(
  X = c(10, 8, 34, 9, 46, 68),
  Y = c(4, 99, 44, 50, 77, 30)
)

# Nuevos datos
new <- data.frame(
  X = c(20, 20, 50),
  Y = c(15, 80, 50)
)

df2 <- df
new2 <- new

print(df)
print(new)

In [ ]:
# Gráfico de dispersión de datos originales y nuevos
plot(df$X, df$Y,
     pch = 19, col = "blue",
     xlab = "X", ylab = "Y",
     main = "Datos originales y nuevas observaciones")
points(new$X, new$Y, pch = 19, col = "black")
legend("topright", legend = c("Originales", "Nuevos"),
       col = c("blue", "black"), pch = 19, bty = "n")

In [ ]:
# Matriz de distancias euclidianas
dist1 <- as.matrix(dist(df, method = "euclidean"))

print(round(dist1, 3))
cat("Distancia media:", mean(dist1), "\n")
print(quantile(dist1, probs = c(0.25, 0.50, 0.75)))

## 2. Mean Shift

Mean Shift es un método basado en densidad. La idea es desplazar iterativamente cada punto hacia zonas de mayor densidad, hasta aproximar una moda local de la distribución de los datos.

A diferencia de K-medias, no se fija directamente el número de grupos. El número de grupos queda inducido por el ancho de banda `h` y por el kernel utilizado.

En R usaremos la función `msClustering()` del paquete `MeanShift`. Esta función espera los datos con variables en filas e individuos en columnas; por esta razón se aplica a `t(df)` y no directamente a `df`.


In [ ]:
# Función auxiliar para asignar nuevas observaciones al centroide empírico más cercano
# Esta no es una predicción propia del paquete; es solo una regla descriptiva posterior.
asignar_por_centroide <- function(newdata, data, labels) {
  centros <- aggregate(data, by = list(Cluster = labels), FUN = mean)
  centros_mat <- as.matrix(centros[, -1, drop = FALSE])
  labels_centros <- centros$Cluster
  new_mat <- as.matrix(newdata)

  pred <- apply(new_mat, 1, function(z) {
    d <- sqrt(rowSums((centros_mat - matrix(z, nrow = nrow(centros_mat),
                                           ncol = length(z), byrow = TRUE))^2))
    labels_centros[which.min(d)]
  })

  return(pred)
}


In [ ]:
# Uso de Mean Shift con el paquete MeanShift
# La función msClustering recibe los datos transpuestos: t(df)
ms <- msClustering(t(df), h = 40)
y_ms <- ms$labels

print(ms)
print(y_ms)
print(table(y_ms))

# Agregar resultado de Mean Shift al dataframe
df2$MS <- y_ms
print(df2)

# Asignar nuevas observaciones al centroide empírico más cercano
new_ms <- asignar_por_centroide(new, df, y_ms)
new2$MS <- new_ms
print(new2)


In [ ]:
# Colores para grupos
cols <- c("red", "blue", "darkgreen", "orange", "purple")
color_ms <- cols[df2$MS]
color_new_ms <- cols[new2$MS]

plot(df$X, df$Y,
     pch = 19, col = color_ms,
     xlab = "X", ylab = "Y",
     main = "Mean Shift")
points(new$X, new$Y, pch = 17, col = "black", cex = 1.4)

centros_ms <- aggregate(df, by = list(Cluster = y_ms), FUN = mean)
points(centros_ms$X, centros_ms$Y, pch = 8, col = "black", cex = 1.8)

legend("topright",
       legend = c("Datos", "Nuevos", "Centroides empíricos"),
       pch = c(19, 17, 8),
       col = c("gray30", "black", "black"),
       bty = "n")


In [ ]:
# Mismo gráfico, coloreando las nuevas observaciones según su asignación posterior
plot(df$X, df$Y,
     pch = 19, col = color_ms,
     xlab = "X", ylab = "Y",
     main = "Mean Shift: asignación de nuevas observaciones")
points(new$X, new$Y, pch = 17, col = color_new_ms, cex = 1.5)
points(centros_ms$X, centros_ms$Y, pch = 8, col = "black", cex = 1.8)


### Ejercicio

Cambie el valor de `h`. Observe que valores pequeños tienden a generar más grupos, mientras que valores grandes tienden a fusionar grupos.

También puede cambiar el kernel:

```r
msClustering(t(df), h = 40, kernel = "gaussianKernel")
msClustering(t(df), h = 40, kernel = "cubicKernel")
msClustering(t(df), h = 40, kernel = "exponentialKernel")
```


## 3. DBSCAN

DBSCAN también es un método basado en densidad. A diferencia de Mean Shift, DBSCAN requiere dos parámetros principales:

- `eps`: radio de vecindad;
- `minPts`: número mínimo de puntos dentro de la vecindad.

Una diferencia importante es que DBSCAN puede detectar observaciones como ruido. En el paquete `dbscan`, las observaciones clasificadas como ruido reciben etiqueta `0`.

In [ ]:
# Aplicar DBSCAN
clust_dbscan <- dbscan(df, eps = 40, minPts = 2)
y_dbs <- clust_dbscan$cluster

print(y_dbs)

df2$DBS <- y_dbs
print(df2)

# Nota: DBSCAN no tiene una función predict natural como K-medias.
# Si se aplica dbscan solo a los nuevos datos, se está ajustando otro modelo.
new_dbs <- dbscan(new, eps = 40, minPts = 2)$cluster
new2$DBS <- new_dbs
print(new2)

In [ ]:
# Colores para DBSCAN: 0 corresponde a ruido
cols_dbs <- c("gray", "red", "blue", "darkgreen", "orange")
# Como las etiquetas parten en 0, sumamos 1 para indexar colores
color_dbs <- cols_dbs[df2$DBS + 1]
color_new_dbs <- cols_dbs[new2$DBS + 1]

plot(df$X, df$Y,
     pch = 19, col = color_dbs,
     xlab = "X", ylab = "Y",
     main = "DBSCAN")
points(new$X, new$Y, pch = 17, col = color_new_dbs, cex = 1.5)
legend("topright", legend = c("Ruido", "Cluster 1", "Cluster 2"),
       col = cols_dbs[1:3], pch = 19, bty = "n")

### Comentario importante

En `sklearn`, DBSCAN tampoco entrega una función `predict` estándar. Ajustar DBSCAN sobre los nuevos datos por separado no equivale a predecir su grupo usando el modelo ajustado con los datos originales. Esto debe interpretarse solo como una ilustración computacional, no como una predicción supervisada.

## 4. Aplicación a la base Iris

Se utiliza la base `iris`, eliminando la variable `Species` durante el ajuste. La especie queda disponible solo como referencia externa para comparar los grupos encontrados.

In [ ]:
# Base de datos iris
data(iris)

X <- iris[, 1:4]
y <- iris$Species

head(X)
summary(X)
boxplot(X, main = "Variables originales")

In [ ]:
# Estandarización
X2 <- as.data.frame(scale(X))
colnames(X2) <- colnames(X)

summary(X2)
boxplot(X2, main = "Variables estandarizadas")

In [ ]:
# Dataframe para almacenar resultados
df_iris <- data.frame(Especie = y)
head(df_iris, 10)

In [ ]:
# Aplicar Mean Shift a Iris
# Se comparan datos originales y datos estandarizados.
# La función msClustering recibe los datos transpuestos: t(X).
iris_ms1 <- msClustering(t(as.matrix(X)))
iris_ms2 <- msClustering(t(as.matrix(X2)))

# También se puede comparar con distintos kernels sobre los datos estandarizados
iris_ms_gauss <- msClustering(t(as.matrix(X2)), kernel = "gaussianKernel")
iris_ms_cubic <- msClustering(t(as.matrix(X2)), kernel = "cubicKernel")
iris_ms_exp <- msClustering(t(as.matrix(X2)), kernel = "exponentialKernel")

y_ms1 <- iris_ms1$labels
y_ms2 <- iris_ms2$labels

df_iris$MS1 <- y_ms1
df_iris$MS2 <- y_ms2

print(table(y_ms1))
print(table(y_ms2))
cat("Número de observaciones con asignación distinta:", sum(y_ms1 != y_ms2), "
")

cat("
Kernels alternativos sobre datos estandarizados:
")
print(table(Gaussian = iris_ms_gauss$labels))
print(table(Cubic = iris_ms_cubic$labels))
print(table(Exponential = iris_ms_exp$labels))


In [ ]:
# Aplicar DBSCAN a Iris
iris_dbs1 <- dbscan(X, eps = 1.5, minPts = 10)
iris_dbs2 <- dbscan(X2, eps = 0.75, minPts = 5)

y_dbs1 <- iris_dbs1$cluster
y_dbs2 <- iris_dbs2$cluster

df_iris$DBS1 <- y_dbs1
df_iris$DBS2 <- y_dbs2

print(table(y_dbs1))
print(table(y_dbs2))
cat("Número de observaciones con asignación distinta:", sum(y_dbs1 != y_dbs2), "\n")

In [ ]:
# Resultados
head(df_iris, 10)

cat("\nTabla: especie vs Mean Shift sin escalar\n")
print(table(Especie = y, Cluster = y_ms1))

cat("\nTabla: especie vs Mean Shift con datos estandarizados\n")
print(table(Especie = y, Cluster = y_ms2))

cat("\nTabla: especie vs DBSCAN sin escalar\n")
print(table(Especie = y, Cluster = y_dbs1))

cat("\nTabla: especie vs DBSCAN con datos estandarizados\n")
print(table(Especie = y, Cluster = y_dbs2))

## 5. Suma de distancias intra-cluster

La siguiente función calcula una medida descriptiva de compacidad a partir de distancias euclidianas dentro de cada grupo.

Para DBSCAN debe tenerse cuidado: la etiqueta `0` corresponde a ruido, por lo que puede excluirse del cálculo.

In [ ]:
sum_intra <- function(X, labels, exclude_noise = FALSE) {
  X <- as.data.frame(X)
  labels <- as.integer(labels)
  grupos <- sort(unique(labels))

  if (exclude_noise) {
    grupos <- grupos[grupos != 0]
  }

  total <- 0
  for (g in grupos) {
    aux <- X[labels == g, , drop = FALSE]
    if (nrow(aux) > 1) {
      D <- as.matrix(dist(aux, method = "euclidean"))
      total <- total + sum(D) / 2
    }
  }
  total
}

cat("W Mean Shift sin escalar:", sum_intra(X, y_ms1), "\n")
cat("W Mean Shift escalado:", sum_intra(X2, y_ms2), "\n")

cat("W DBSCAN sin escalar, incluyendo ruido:", sum_intra(X, y_dbs1), "\n")
cat("W DBSCAN escalado, incluyendo ruido:", sum_intra(X2, y_dbs2), "\n")

cat("W DBSCAN sin escalar, excluyendo ruido:", sum_intra(X, y_dbs1, exclude_noise = TRUE), "\n")
cat("W DBSCAN escalado, excluyendo ruido:", sum_intra(X2, y_dbs2, exclude_noise = TRUE), "\n")

## 6. Visualización mediante componentes principales

Como `iris` tiene cuatro variables, se usa PCA para visualizar las observaciones en dos dimensiones. El PCA se calcula sobre variables estandarizadas.

In [ ]:
# Componentes principales
pca1 <- prcomp(X, center = TRUE, scale. = TRUE)
pca2 <- prcomp(X2, center = TRUE, scale. = FALSE)

cat("Porcentaje de varianza explicada - PCA sobre X con scale.=TRUE\n")
print(round(summary(pca1)$importance[2, ] * 100, 2))
print(round(summary(pca1)$importance[3, ] * 100, 2))

cat("\nPorcentaje de varianza explicada - PCA sobre X2 ya escalada\n")
print(round(summary(pca2)$importance[2, ] * 100, 2))
print(round(summary(pca2)$importance[3, ] * 100, 2))

cat("\nVectores propios\n")
print(round(pca1$rotation, 3))

In [ ]:
# Proyección de las observaciones
proyecciones <- as.data.frame(pca1$x)
head(proyecciones, 10)

In [ ]:
# Visualización con las primeras dos componentes principales: especies originales
cols_species <- c("red", "blue", "darkgreen")
plot(proyecciones$PC1, proyecciones$PC2,
     pch = 19, col = cols_species[as.integer(y)],
     xlab = "Componente 1", ylab = "Componente 2",
     main = "Iris: especies originales")
legend("topright", legend = levels(y),
       col = cols_species, pch = 19, bty = "n")

In [ ]:
# Función auxiliar para graficar clusters sobre PCA
plot_clusters_pca <- function(proy, labels, titulo) {
  labels <- as.integer(labels)
  unique_labels <- sort(unique(labels))
  pal <- c("gray", "red", "blue", "darkgreen", "orange", "purple", "brown")

  # Si aparece etiqueta 0, se interpreta como ruido y queda gris.
  col_idx <- ifelse(labels == 0, 1, labels + 1)

  plot(proy$PC1, proy$PC2,
       pch = 19, col = pal[col_idx],
       xlab = "Componente 1", ylab = "Componente 2",
       main = titulo)
}

plot_clusters_pca(proyecciones, y_ms1, "Iris con Mean Shift sin escalar")
plot_clusters_pca(proyecciones, y_ms2, "Iris con Mean Shift escalado")
plot_clusters_pca(proyecciones, y_dbs1, "Iris con DBSCAN sin escalar")
plot_clusters_pca(proyecciones, y_dbs2, "Iris con DBSCAN escalado")

## 7. Comentarios finales

- Mean Shift y DBSCAN son métodos basados en densidad.
- En ambos métodos, los resultados pueden cambiar considerablemente al modificar los hiperparámetros.
- En DBSCAN, la etiqueta `0` representa ruido.
- El escalamiento es importante cuando las variables están en distintas unidades o escalas.
- PCA permite visualizar los grupos en dos dimensiones, pero no reemplaza el análisis en el espacio original de variables.

### Preguntas para discusión

1. ¿Qué ocurre con Mean Shift al aumentar o disminuir `bandwidth`?
2. ¿Qué ocurre con DBSCAN al modificar `eps` y `minPts`?
3. ¿Tiene sentido comparar DBSCAN usando una suma intra-cluster si existe ruido?
4. ¿Qué método utilizaría para estos datos y por qué?